# Exercise 02 — Optimizing Synaptic Input Computation

**Module 02 | Estimated time: 1–2 hours**

---

## Background

Computing the total synaptic input to each neuron is the most computationally intensive step in a dense neural network simulation:

$$I_j = \sum_{i=0}^{N-1} W_{ij} \cdot s_i \quad \text{for all } j$$

This is a matrix-vector product **I = W · s** where:
- $W$ is the $N \times N$ synaptic weight matrix
- $s$ is the $N$-vector of pre-synaptic firing rates
- $I$ is the $N$-vector of total input currents

## Task

You will optimize this computation in four steps:

1. **Baseline:** naive kernel (one thread per output neuron, loops over all inputs)
2. **Optimization 1:** ensure coalesced memory access
3. **Optimization 2:** use shared memory tiling
4. **Benchmark:** compare all versions and plot speedup
5. *(Challenge)* Add constant memory for a conductance scaling factor

In [ ]:
!nvidia-smi

## Step 1: Naive Kernel

Each thread computes one output neuron's total input.
Thread j computes: `I[j] = sum over i of W[i*N+j] * s[i]`

In [ ]:
%%writefile synaptic_input.cu
#include <stdio.h>
#include <stdlib.h>
#include <math.h>
#include <cuda_runtime.h>

#define CUDA_CHECK(call) do { cudaError_t e=(call); if(e!=cudaSuccess){ \
    fprintf(stderr,"CUDA: %s\n",cudaGetErrorString(e)); exit(1);}} while(0)

#define TILE 16

// ── Kernel 1: Naive ──────────────────────────────────────────────────────────
// Thread j computes I[j] = sum_i W[i,j] * s[i]
// W is stored row-major: W[i,j] = W_data[i*N + j]
__global__ void matvec_naive(const float* W, const float* s, float* I, int N) {
    int j = blockIdx.x * blockDim.x + threadIdx.x;   // output neuron
    if (j >= N) return;
    float sum = 0.0f;
    for (int i = 0; i < N; i++) {
        // TODO: Is this access to W coalesced? Why or why not?
        sum += W[i * N + j] * s[i];
    }
    I[j] = sum;
}

// ── Kernel 2: Coalesced ──────────────────────────────────────────────────────
// TODO: Rewrite so thread j reads W[j, i] (row-major, row j)
// This requires storing W transposed (W^T[j,i] = W[i,j])
// or computing I^T = s^T * W instead.
// Hint: thread j reads row j of W^T: Wt[j*N + i] for all i.
__global__ void matvec_coalesced(const float* Wt, const float* s, float* I, int N) {
    int j = ???;
    if (j >= N) return;
    float sum = 0.0f;
    for (int i = 0; i < N; i++) {
        sum += Wt[???] * s[i];   // TODO: fill in the Wt index
    }
    I[j] = sum;
}

// ── Kernel 3: Tiled with shared memory ──────────────────────────────────────
// TODO: Tile the matvec using shared memory.
// Strategy: each block processes a tile of s and accumulates into sum.
// 1. Load s[tile_start ... tile_start+TILE-1] into __shared__ float s_tile[TILE]
// 2. Load Wt[j, tile_start ... tile_end] into __shared__ float w_tile[TILE]
// 3. Compute partial dot product from shared memory
// 4. Advance to next tile
__global__ void matvec_tiled(const float* Wt, const float* s, float* I, int N) {
    __shared__ float s_tile[TILE];
    // TODO: add w_tile declaration

    int j = blockIdx.x * blockDim.x + threadIdx.x;
    float sum = 0.0f;

    for (int t = 0; t < (N + TILE - 1) / TILE; t++) {
        // TODO: cooperatively load s_tile
        // (Only threadIdx.x < TILE should load, others idle)
        if (threadIdx.x < TILE) {
            int idx = t * TILE + threadIdx.x;
            s_tile[threadIdx.x] = (idx < N) ? s[idx] : 0.0f;
        }
        __syncthreads();

        // TODO: load w_tile and compute partial dot product
        ???

        __syncthreads();
    }

    if (j < N) I[j] = sum;
}

// ── CPU reference ─────────────────────────────────────────────────────────────
void matvec_cpu(const float* W, const float* s, float* I, int N) {
    for (int j = 0; j < N; j++) {
        float sum = 0;
        for (int i = 0; i < N; i++) sum += W[i*N+j] * s[i];
        I[j] = sum;
    }
}

float max_err(const float* a, const float* b, int n) {
    float e = 0;
    for (int i = 0; i < n; i++) { float d=fabsf(a[i]-b[i]); if(d>e) e=d; }
    return e;
}

int main() {
    const int N = 1024;
    size_t mat_bytes = (size_t)N*N*sizeof(float);
    size_t vec_bytes = N * sizeof(float);

    float* h_W  = (float*)malloc(mat_bytes);
    float* h_Wt = (float*)malloc(mat_bytes);   // transposed W
    float* h_s  = (float*)malloc(vec_bytes);
    float* h_I  = (float*)malloc(vec_bytes);
    float* h_ref= (float*)malloc(vec_bytes);

    srand(42);
    for (int i = 0; i < N*N; i++) h_W[i] = (float)rand()/RAND_MAX * 0.01f;
    for (int j = 0; j < N; j++) h_s[j] = (float)rand()/RAND_MAX;
    // Transpose W
    for (int i = 0; i < N; i++)
        for (int j = 0; j < N; j++)
            h_Wt[j*N+i] = h_W[i*N+j];

    // CPU reference
    matvec_cpu(h_W, h_s, h_ref, N);

    float *d_W, *d_Wt, *d_s, *d_I;
    CUDA_CHECK(cudaMalloc(&d_W,  mat_bytes));
    CUDA_CHECK(cudaMalloc(&d_Wt, mat_bytes));
    CUDA_CHECK(cudaMalloc(&d_s,  vec_bytes));
    CUDA_CHECK(cudaMalloc(&d_I,  vec_bytes));
    CUDA_CHECK(cudaMemcpy(d_W,  h_W,  mat_bytes, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_Wt, h_Wt, mat_bytes, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_s,  h_s,  vec_bytes, cudaMemcpyHostToDevice));

    int threads = 256, blocks = (N + threads - 1) / threads;
    cudaEvent_t t0, t1;
    CUDA_CHECK(cudaEventCreate(&t0)); CUDA_CHECK(cudaEventCreate(&t1));
    float ms;

    printf("%-18s  %-10s  %-10s  %-8s\n", "Kernel", "Time (ms)", "BW (GB/s)", "Status");
    printf("%-18s  %-10s  %-10s  %-8s\n", "------", "---------", "---------", "------");

    #define BENCH(label, kernel_call) \
        CUDA_CHECK(cudaEventRecord(t0)); \
        kernel_call; \
        CUDA_CHECK(cudaEventRecord(t1)); \
        CUDA_CHECK(cudaEventSynchronize(t1)); \
        CUDA_CHECK(cudaEventElapsedTime(&ms, t0, t1)); \
        CUDA_CHECK(cudaMemcpy(h_I, d_I, vec_bytes, cudaMemcpyDeviceToHost)); \
        printf("%-18s  %-10.3f  %-10.1f  %-8s\n", label, ms, \
               (mat_bytes + vec_bytes * 2.0) / (ms * 1e-3) / 1e9, \
               max_err(h_I, h_ref, N) < 1e-3f ? "PASS" : "FAIL");

    BENCH("Naive",     matvec_naive<<<blocks, threads>>>(d_W, d_s, d_I, N))
    BENCH("Coalesced", matvec_coalesced<<<blocks, threads>>>(d_Wt, d_s, d_I, N))
    BENCH("Tiled",     matvec_tiled<<<blocks, threads>>>(d_Wt, d_s, d_I, N))

    CUDA_CHECK(cudaEventDestroy(t0)); CUDA_CHECK(cudaEventDestroy(t1));
    cudaFree(d_W); cudaFree(d_Wt); cudaFree(d_s); cudaFree(d_I);
    free(h_W); free(h_Wt); free(h_s); free(h_I); free(h_ref);
    return 0;
}

In [ ]:
!nvcc -O2 -o synaptic_input synaptic_input.cu -lm && ./synaptic_input

## Step 4: Benchmark Across Network Sizes

In [ ]:
import subprocess
import matplotlib.pyplot as plt
import numpy as np

# TODO: Modify synaptic_input.cu to accept N as a command-line argument,
# then sweep N and plot speedup of tiled vs naive.
# Hint: use argc/argv and atoi().

print("Implement the benchmark sweep here after modifying the .cu file.")

## Challenge: Constant Memory for Conductance Scaling

Add a `__constant__ float c_g_syn;` (synaptic conductance) and multiply the final result:
```
I[j] = c_g_syn * (sum_i W[i,j] * s[i])
```
This avoids passing the conductance as a kernel parameter and demonstrates constant memory caching.

---
Check your work against [ex02_solution.ipynb](ex02_solution.ipynb).